# RECIST response & the ORR → OS surrogate

**NOT FOR CLINICAL USE.** Population/trial-level response rates only; no individual response probability, no therapy ranking. Executed in CI (nbmake).

The objective response rate (ORR) is the dominant phase-2 go/no-go endpoint, and a famously contested OS surrogate — drugs with high response rates routinely fail to extend survival. Onkos had OS/PFS but no response endpoint. This adds it, and because a model's ORR *and* OS come from the **same simulated trial**, it makes the ORR→OS relationship measured. The sharp result: whether ORR faithfully ranks survival is **conditional on the survival mechanism**. See `docs/specs/research/recist-orr-surrogate.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.response import best_response, objective_response_rate, response_vs_survival

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
print(f'{len(ds)} records | onkos {onkos.__version__}')

## RECIST best response → population rates

Best overall response is classified per RECIST 1.1 from the observed baseline (CR ≥95% shrink, PR ≥30%, PD ≥20% regrowth from nadir with no PR, else SD). ORR = P(CR or PR), DCR = P(CR, PR, or SD), over the stored inter-patient variability (the IIV ensemble).

In [ ]:
# classification boundaries
t = np.linspace(0, 52, 105)
print('exactly -30% ->', best_response(t, np.linspace(100, 70, 105)))   # PR
print('only  -29% ->', best_response(t, np.linspace(100, 71, 105)))     # SD
print('-10% then +60% regrowth ->', best_response(t, np.concatenate([np.linspace(100,90,50), np.linspace(90,144,55)])))  # PD

print()
for rid in ['drug_effect.norton_simon.nsclc','resistance.claret_2009.tgi','resistance.nsclc_first_line.two_population','tgi_metrics.wang_2009.biexponential']:
    rr = objective_response_rate(ds, rid, context=ctx, n=400)
    d = rr.distribution
    print(f"{rid.split('.')[1]:<22} ORR={rr.orr:.2f} DCR={rr.dcr:.2f}  CR/PR/SD/PD = "
          f"{d['CR']:.2f}/{d['PR']:.2f}/{d['SD']:.2f}/{d['PD']:.2f}")

## The ORR → OS surrogate is conditional on the survival mechanism

ORR and the week-8 survival surrogate are *both* shrinkage-based, so ORR ranks OS perfectly under the week-8 link. But under the tail-sensitive k_g link (v0.25), a deep early responder that regrows fast has a high ORR and a SHORT OS — ORR inverts. `response_vs_survival` counts the discordant model pairs.

In [ ]:
for link, name in [(None, 'week-8 (default)'), ('survival_link.nsclc_os_growth_rate', 'k_g (tail-sensitive)')]:
    rs = response_vs_survival(ds, context=ctx, survival_link=link, n=400)
    print(f'OS link: {name}')
    for r in sorted(rs.rows, key=lambda x: -x['orr']):
        print(f"   {r['record_id'].split('.')[1]:<22} ORR {r['orr']:.2f}  mOS {r['median_os_weeks']:.0f}")
    print(f"   -> discordant {rs.discordant_pairs}/{rs.total_pairs} (frac {rs.discordant_fraction:.2f}); ORR predicts OS? {rs.orr_predicts_os}\n")

# Under week-8 ORR is a faithful surrogate; under k_g it is not.
assert response_vs_survival(ds, context=ctx, n=400).orr_predicts_os
assert not response_vs_survival(ds, context=ctx, survival_link='survival_link.nsclc_os_growth_rate', n=400).orr_predicts_os

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for link, color, name in [(None, '#2b6cb0', 'week-8'), ('survival_link.nsclc_os_growth_rate', '#c53030', 'k_g')]:
    rs = response_vs_survival(ds, context=ctx, survival_link=link, n=400)
    rows = sorted(rs.rows, key=lambda x: x['orr'])
    ax.plot([r['orr'] for r in rows], [r['median_os_weeks'] for r in rows], 'o-', color=color,
            label=f'{name} link (discordant {rs.discordant_pairs}/{rs.total_pairs})')
ax.set(xlabel='ORR', ylabel='median OS (weeks)', title='ORR vs OS — faithful under week-8, inverted under k_g')
ax.legend(fontsize=8)
plt.show()

## Guardrails

Population/trial-level rates only — no individual response probability. The rates carry the chain's propagated tier (out-of-context → D), and the surrogate result is a statement about *models*, never a therapy ranking. Onkos shows the discordance under each survival assumption; it does not certify ORR as a valid or invalid surrogate.

In [ ]:
rr = objective_response_rate(ds, 'resistance.claret_2009.tgi', context=ctx, n=200)
assert np.isclose(sum(rr.distribution.values()), 1.0) and 0 <= rr.orr <= rr.dcr <= 1
out = objective_response_rate(ds, 'resistance.claret_2009.tgi', context={'tumor_type':'melanoma','line':'first'}, n=80)
print('transported tier:', out.tier)
assert out.tier == 'D'
d = rr.to_dict()
assert d['NOT_FOR_CLINICAL_USE'] is True and 'PROHIBITED' in d['onkos:clinicalUse']
print('guardrails OK | ORR', round(rr.orr, 2), '| simplex + tier passthrough verified')